# Notebook 3: Clean and Merge Tree Census Data

This is the third notebook in the sequence for the HRS Botany project.

In it we will clean and merge the two tree census datasets (FERP and OFO), in preparation for gridding by EnMAP pixels.


__Notebook Inputs:__
- OFO Trees Dataset (_ofo_ground-reference_trees.gpkg_)
- FERP Data (_FERP123merged_20231029.csv_)

__Notebook Outputs:__
- Cleaned and merged Tree Dataset (_data/trees.gpkg_)


In [38]:
# imports

import pandas as pd
import geopandas as gpd

from hrs_botany.data_utils import load_ferp_species_table

## OFO Trees Dataset

In [3]:
ofo_trees_df = gpd.read_file('../../data/ofo/ofo_ground-reference_trees.gpkg')
plots_df = gpd.read_file('./data/plots.csv')

Let's restrict the ofo trees dataframe to the plots we selected in notebook 0.

In [4]:
ofo_plot_l = list(plots_df.plot_id[:-3])

ofo_trees_df = ofo_trees_df[ofo_trees_df.plot_id.isin(ofo_plot_l)].reset_index(drop=True)

In [5]:
print('Number of records across all selected OFO plots:', len(ofo_trees_df))

Number of records across all selected OFO plots: 34471


The OFO tree dataset specifies the tree species by its USDA PLANTS symbol. To standardize which species code we are using across datasets, we will convert these to GBIF Taxon IDs. 

First, we will convert the species_code to scientific name.

In [6]:
# 1. Load the USDA PLANTS symbols file
usda = pd.read_csv(
    "./data/usda/usda_plants_codes.txt",
    low_memory=False,
    dtype=str,
    skipinitialspace=True
)

# 2. Keep only the primary (non‐synonym) rows
primary = usda[usda["Synonym Symbol"].isna() | (usda["Synonym Symbol"] == "")].copy()

# 3. Optional: trim whitespace
for col in ["Symbol","Scientific Name with Author","Common Name","Family"]:
    primary[col] = primary[col].str.strip()

# 4. Build a lookup table
lookup = primary[[
    "Symbol",
    "Scientific Name with Author",
    "Common Name",
    "Family"
]].rename(columns={
    "Symbol":                      "species_code",
    "Scientific Name with Author": "scientific_name",
    "Common Name":                 "common_name",
    "Family":                      "family"
})

# 5. First pass merge into your tree_df
ofo_trees_df = ofo_trees_df.merge(
    lookup,
    on="species_code",
    how="left"
)

# 6. Inspect any unmatched codes
missing = ofo_trees_df[ofo_trees_df["scientific_name"].isna()]["species_code"].unique()
if len(missing):
    print("No USDA match for:", missing)
else:
    print("All codes matched!")

# 7. Hand-curated scientific_name overrides for leftover species_codes:
manual_matches = {
    "RHPU":    {"scientific_name": "Frangula purshiana"},
    "ARCPRI":  {"scientific_name": "Arctostaphylos pringlei"},
    "UNKSNAG": {"scientific_name": "Tsuga mertensiana"},
    "FRACAL":  {"scientific_name": "Frangula californica"},
    "AB":      {"scientific_name": "Fagus grandifolia"},
    "PIPJ":    {"scientific_name": "Picea pungens"},
    "CEAPAL":  {"scientific_name": "Ceanothus palmeri"},
    "QUEXMO":  {"scientific_name": "Quercus xmorehus"},
    "RHOOCC":  {"scientific_name": "Rhododendron occidentale"},
    "ARCGLA":  {"scientific_name": "Arctostaphylos glandulosa"},
    "PI":      {"scientific_name": "Picea spp."},
    "LONSUB":  {"scientific_name": "Lonicera subspicata"},
    "QUEV":    {"scientific_name": "Quercus spp."},
    "ARCPUN":  {"scientific_name": "Arctostaphylos pungens"},
    "UNK":     {"scientific_name": "Unknown"},
}

# 8. Build a tiny DataFrame of those overrides
manual_df = (
    pd.DataFrame.from_dict(manual_matches, orient="index")
      .reset_index()
      .rename(columns={"index": "species_code"})
)

# 9. Merge into ofo_trees_df, filling only where the lookup left blanks
ofo_trees_df = (
    ofo_trees_df
    .merge(manual_df, on="species_code", how="left", suffixes=("", "_manual"))
)

# 10. For scientific_name, fall back to the manual value if it was missing
ofo_trees_df["scientific_name"] = ofo_trees_df["scientific_name"].fillna(
    ofo_trees_df["scientific_name_manual"]
)
ofo_trees_df.drop(columns="scientific_name_manual", inplace=True)

# 11. Final sanity check
still_missing = ofo_trees_df.loc[
    ofo_trees_df["scientific_name"].isna(), "species_code"
].unique()
if len(still_missing):
    print("Still no scientific_name for:", still_missing)
else:
    print("All species_codes now have scientific_name!")


# Now tree_df has:
#  - usda_symbol
#  - scientific_name
#  - common_name
#  - family
#  - fia_code


No USDA match for: ['PI' 'UNKSNAG' 'QUEV' 'PIPJ' 'AB' 'UNK' 'ARCPRI' 'FRACAL' 'CEAPAL'
 'RHOOCC' 'ARCGLA' 'LONSUB' 'ARCPUN' 'QUEXMO' 'RHPU']
All species_codes now have scientific_name!


We will now join in the GBIF Taxon ID. 

And there are a few typos in the scientific names: we will now manually fix them.

In [7]:
import pandas as pd
from pygbif import species
import time

# 1. Copy and normalize your scientific names

ofo_trees_df['sciname'] = (
    ofo_trees_df['scientific_name']
      .str.strip()
      .str.split()
      .str[:2]
      .str.join(' ')
      .str.lower()
)

# 2. Prepare a cache dict + manual overrides for the few bad names
taxon_cache = {}
name_overrides = {
    'quercus ×deamii': 'quercus deamii',  # drop the hybrid sign
    'salix l.':          'Salix L.',         # look up the genus  
    'unknown':           None             # skip entirely
}

# 3. Iterate unique names and query GBIF (with overrides)
for name in ofo_trees_df['sciname'].dropna().unique():
    if name in name_overrides:
        override = name_overrides[name]
        # if override is None, we know it’s “unknown” → leave as None
        if override is None:
            taxon_cache[name] = None
            continue
        lookup_name = override
    else:
        lookup_name = name

    try:
        res = species.name_backbone(name=lookup_name)
        key = res.get('usageKey') or None
    except Exception:
        key = None

    taxon_cache[name] = key
    time.sleep(0.1)  # be polite to GBIF

# 4. Map back onto your DataFrame
ofo_trees_df['gbif_taxon_key'] = ofo_trees_df['sciname'].map(taxon_cache)

# 5. Check what still didn’t resolve
unmapped = sorted(x for x, k in taxon_cache.items() if k is None)
print(f"Unmapped names ({len(unmapped)}):", unmapped)


Unmapped names (1): ['unknown']


In [8]:
live_ofo_tree_df = ofo_trees_df[(ofo_trees_df.live_dead == 'L') & (ofo_trees_df.growth_form == 'tree')]
print('Number of live trees:', len(live_ofo_tree_df))
print('Percent live trees missing height:', sum(live_ofo_tree_df.height.isna())/len(live_ofo_tree_df))
print('Percent live trees missing dbh:', sum(live_ofo_tree_df.dbh.isna())/len(live_ofo_tree_df))
print('Percent live trees missing crown_position:', sum(live_ofo_tree_df.crown_position.isna())/len(live_ofo_tree_df))
print('Percent live trees missing crown_ratio:', sum(live_ofo_tree_df.crown_ratio.isna())/len(live_ofo_tree_df))
print('Percent live trees missing crown_ratio_compacted:', sum(live_ofo_tree_df.crown_ratio_compacted.isna())/len(live_ofo_tree_df))
print('Percent live trees missing height_to_crown:', sum(live_ofo_tree_df.height_to_crown.isna())/len(live_ofo_tree_df))
print('Percent live trees missing ohvis:', sum(live_ofo_tree_df.ohvis.isna())/len(live_ofo_tree_df))
print('Percent live trees missing crown_width_1:', sum(live_ofo_tree_df.crown_width_1.isna())/len(live_ofo_tree_df))
print('Percent live trees missing crown_width_2:', sum(live_ofo_tree_df.crown_width_2.isna())/len(live_ofo_tree_df))

Number of live trees: 27860
Percent live trees missing height: 0.6198851399856425
Percent live trees missing dbh: 0.059547738693467335
Percent live trees missing crown_position: 0.7582555635319455
Percent live trees missing crown_ratio: 1.0
Percent live trees missing crown_ratio_compacted: 1.0
Percent live trees missing height_to_crown: 0.9917803302225413
Percent live trees missing ohvis: 0.7582555635319455
Percent live trees missing crown_width_1: 0.9917803302225413
Percent live trees missing crown_width_2: 0.9917803302225413


In [9]:
ofo_cols = ['plot_id', 'gbif_taxon_key', 'geometry', 'scientific_name', 'common_name', 'height', 'dbh', 'crown_position']

ofo_trees_df = ofo_trees_df[ofo_cols]


In [10]:
ofo_trees_df

,plot_id,gbif_taxon_key,geometry,scientific_name,common_name,height,dbh,crown_position
0,0068,2685580.0,POINT (-120.08706 38.96704),Abies concolor (Gord. & Glend.) Lindl. ex Hild...,white fir,26.2,51.8,NaN
1,0068,2683936.0,POINT (-120.08698 38.96712),Calocedrus decurrens (Torr.) Florin,incense cedar,4.5,13.9,NaN
2,0068,2683936.0,POINT (-120.087 38.96696),Calocedrus decurrens (Torr.) Florin,incense cedar,19.4,43.2,NaN
3,0068,2683936.0,POINT (-120.08705 38.967),Calocedrus decurrens (Torr.) Florin,incense cedar,23.3,67.3,NaN
4,0068,2683936.0,POINT (-120.08709 38.967),Calocedrus decurrens (Torr.) Florin,incense cedar,6.4,19.9,NaN
...,...,...,...,...,...,...,...,...
34466,0087,2685580.0,POINT (-119.02263 36.96469),Abies concolor (Gord. & Glend.) Lindl. ex Hild...,white fir,3.0,5.5,NaN
34467,0087,2683936.0,POINT (-119.02267 36.96455),Calocedrus decurrens (Torr.) Florin,incense cedar,3.0,5.9,5.0
34468,0087,2683936.0,POINT (-119.02261 36.96447),Calocedrus decurrens (Torr.) Florin,incense cedar,NaN,13.1,NaN
34469,0087,8286637.0,POINT (-119.02267 36.96427),Frangula purshiana,NaN,NaN,NaN,NaN


## FERP Trees Dataset

Now we will load and prepare the FERP tree dataset.

In [11]:
ferp_path = '../../data/ferp/geoforest/doi_10_5061_dryad_6q573n64s__v20240129/FERP123merged_20231029.csv'

# Load the dataset
ferp_trees_df = pd.read_csv(ferp_path, low_memory=False)

In [12]:
print('survey 1 status:', ferp_trees_df.status1.value_counts())
print('survey 2 status:', ferp_trees_df.status2.value_counts())
print('survey 3 status:', ferp_trees_df.status3.value_counts())


survey 1 status: status1
living    11646
Name: count, dtype: int64
survey 2 status: status2
living    39821
dead       1006
nodata       32
Name: count, dtype: int64
survey 3 status: status3
living      36849
dead        10496
nodata        192
living\t        1
Name: count, dtype: int64


In preparing the FERP trees dataset, it is important to account for how the data encapsulates information across all three surveys of the FERP plot.

Each row in the FERP trees dataset represents a single tree. For each row, there are both static and survey variables. Static variables are those that do not change across the individual surveys, such as species, geographical position, and quadrat. Survey variables are those that are collected in each survey, and are subject to change over time such as height, dbh, live/dead, etc. Survey variables in the FERP dataset usually have 3 separate columns – 1 for each survey.

The FERP plot censuses took place in 2007, 2012, and 2017.

Now since all the EnMAP images are from 2022 and later, we can simplify our data to use the most up to date survey information. This means we can rely on the most recent 2017 survey data for the majority of the trees. Exceptions include trees that have NaNs in their 3rd survey columns, which either means that tree wasn't remeasured in the most recent survey, or it had been declared dead in a previous survey and skipped in subsequent surveys.

With this in mind, we will simplify our columns.

In [13]:
import pandas as pd

# 1. Columns
id_cols = ["quadrat", "tag", "stemtag", "stemtag1"]
static_cols = [
    "code6", "east_m", "north_m", "east_UTM", "north_UTM", 
    "first_census", "irreg_dsh", "hom_m", "code6fix", "locfix"
]
grouped_vars = {
    "dsh_mm": ["dsh3_mm", "dsh2_mm", "dsh1_mm"],
    "dshm_mm": [None, None, "dsh1m_mm"],
    "status": ["status3", "status2", "status1"],
    "condition": ["condition3", "condition2", "condition1"],
    "date": ["date3", "date2", "date1"],
    "multi": ["multi3", "multi2", "multi1"],
    "basalarea_m2": [None, None, "basalarea1_m2"],
    "notes": ["notes3", "notes2", None],
}
census_map = {0: "FERP3", 1: "FERP2", 2: "FERP1"}

# 2. Remove stems ever declared dead
is_dead = ferp_trees_df[["status1", "status2", "status3"]].apply(
    lambda row: row.astype(str).str.lower().eq("dead").any(), axis=1
)
ferp_filtered = ferp_trees_df[~is_dead].copy()

# 3. Determine latest usable survey per row
def pick_survey(row):
    for i, suffix in enumerate(["3", "2", "1"]):
        col = f"dsh{suffix}_mm"
        if pd.notna(row.get(col)):
            return f"FERP{suffix}"
    return pd.NA

ferp_filtered["survey_used"] = ferp_filtered.apply(pick_survey, axis=1)

# 4. Drop stems with no usable census data
ferp_filtered = ferp_filtered[ferp_filtered["survey_used"].notna()].copy()

# 5. Build final dataframe
result_df = ferp_filtered[id_cols + static_cols + ["survey_used"]].copy()

# 6. Pull values from selected survey
for varname, census_cols in grouped_vars.items():
    values = []
    for i, row in ferp_filtered.iterrows():
        survey = row["survey_used"]
        idx = int(str(survey)[-1]) - 1
        col = census_cols[idx]
        val = row[col] if col is not None else pd.NA
        values.append(val)
    result_df[varname] = values


In [14]:
result_df.columns

Index(['quadrat', 'tag', 'stemtag', 'stemtag1', 'code6', 'east_m', 'north_m',
       'east_UTM', 'north_UTM', 'first_census', 'irreg_dsh', 'hom_m',
       'code6fix', 'locfix', 'survey_used', 'dsh_mm', 'dshm_mm', 'status',
       'condition', 'date', 'multi', 'basalarea_m2', 'notes'],
      dtype='object')

In [15]:
ferp_cols = ['code6', 'east_UTM', 'north_UTM', 'survey_used', 'dsh_mm', 'dshm_mm', 'status', 'condition', 'date', 'multi', 'basalarea_m2']

ferp_trees_df = result_df[ferp_cols]

In [16]:
ferp_trees_df

,code6,east_UTM,north_UTM,survey_used,dsh_mm,dshm_mm,status,condition,date,multi,basalarea_m2
0,QUERPA,582309.51,4096655.62,FERP3,31.0,NaN,living,NaN,12/8/06,NaN,0.000755
1,PSEUME,582307.45,4096655.65,FERP3,378.0,NaN,living,NaN,12/8/06,NaN,0.112221
2,QUERPA,582307.51,4096656.25,FERP3,20.0,NaN,living,NaN,12/8/06,NaN,0.000314
3,SEQUSE,582311.77,4096662.27,FERP3,1420.0,NaN,living,NaN,12/8/06,NaN,1.583677
5,QUERPA,582314.71,4096667.18,FERP3,NaN,NaN,NaN,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...
51011,PSEUME,582677.22,4096891.09,FERP2,34.0,<NA>,living,NaN,7/23/14,NaN,<NA>
51012,QUERPA,582671.76,4096867.51,FERP2,44.0,<NA>,living,NaN,7/25/14,NaN,<NA>
51013,SEQUSE,582728.73,4096926.40,FERP2,11.0,<NA>,living,leaning,8/12/14,NaN,<NA>
51014,LITHDE,582651.49,4096603.23,FERP3,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [17]:
ferp_trees_df[ferp_trees_df.status == 'living']

,code6,east_UTM,north_UTM,survey_used,dsh_mm,dshm_mm,status,condition,date,multi,basalarea_m2
0,QUERPA,582309.51,4096655.62,FERP3,31.0,NaN,living,NaN,12/8/06,NaN,0.000755
1,PSEUME,582307.45,4096655.65,FERP3,378.0,NaN,living,NaN,12/8/06,NaN,0.112221
2,QUERPA,582307.51,4096656.25,FERP3,20.0,NaN,living,NaN,12/8/06,NaN,0.000314
3,SEQUSE,582311.77,4096662.27,FERP3,1420.0,NaN,living,NaN,12/8/06,NaN,1.583677
6,LITHDE,582314.46,4096662.70,FERP3,31.0,NaN,living,NaN,12/8/06,NaN,0.000755
...,...,...,...,...,...,...,...,...,...,...,...
51009,PSEUME,582677.05,4096891.65,FERP2,26.0,<NA>,living,NaN,7/23/14,NaN,<NA>
51010,PSEUME,582677.29,4096891.38,FERP2,14.0,<NA>,living,leaning,7/23/14,NaN,<NA>
51011,PSEUME,582677.22,4096891.09,FERP2,34.0,<NA>,living,NaN,7/23/14,NaN,<NA>
51012,QUERPA,582671.76,4096867.51,FERP2,44.0,<NA>,living,NaN,7/25/14,NaN,<NA>


In [18]:
ferp_trees_df.isna().describe()

,code6,east_UTM,north_UTM,survey_used,dsh_mm,dshm_mm,status,condition,date,multi,basalarea_m2
count,36958,36958,36958,36958,36958,36958,36958,36958,36958,36958,36958
unique,1,1,1,1,2,1,2,2,2,2,2
top,False,False,False,False,True,True,True,True,True,True,True
freq,36958,36958,36958,36958,31735,36958,31706,36577,31735,36084,31900


In [19]:
ferp_species_path = './data/ferp/ferp_tree_species.txt'

df_species = load_ferp_species_table(ferp_species_path)

df_species.head(3)

,Scientific name,Common name,Code,Family,Related,Genus,Specific epithet,Author
0,Acer macrophyllum Pursh,Big-leaf maple,ACERMA,Sapindaceae,,Acer,macrophyllum,Pursh
1,Adenostoma fasciculatum Hook. & Arn.,Chamise,ADENFA,Rosaceae,,Adenostoma,fasciculatum,Hook. & Arn.
2,Arbutus menziesii Pursh,Madrone,ARBUME,Ericaceae,,Arbutus,menziesii,Pursh


In [20]:
common_name_d = df_species.set_index('Code')['Common name'].to_dict()
specific_epithet_d = df_species.set_index('Code')['Specific epithet'].to_dict()
genus_d = df_species.set_index('Code')['Genus'].to_dict()

ferp_trees_df['common_name'] = ferp_trees_df['code6'].map(common_name_d)
ferp_trees_df['scientific_name'] = ferp_trees_df['code6'].map(genus_d) + ' ' + ferp_trees_df['code6'].map(specific_epithet_d)

/var/folders/qx/bpj16cl90cq20swjd4j79cdh0000gn/T/ipykernel_56129/904045117.py:5: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  ferp_trees_df['common_name'] = ferp_trees_df['code6'].map(common_name_d)
/var/folders/qx/bpj16cl90cq20swjd4j79cdh0000gn/T/ipykernel_56129/904045117.py:6: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  ferp_trees_df['scientific_name'] = ferp_trees_df['code6'].map(genus_d) + ' ' + ferp_trees_df['code6'].map(specific_epithet_d)


In [21]:
ferp_trees_df

,code6,east_UTM,north_UTM,survey_used,dsh_mm,dshm_mm,status,condition,date,multi,basalarea_m2,common_name,scientific_name
0,QUERPA,582309.51,4096655.62,FERP3,31.0,NaN,living,NaN,12/8/06,NaN,0.000755,Shreve’s oak,Quercus parvula
1,PSEUME,582307.45,4096655.65,FERP3,378.0,NaN,living,NaN,12/8/06,NaN,0.112221,Douglas-fir,Pseudotsuga menziesii
2,QUERPA,582307.51,4096656.25,FERP3,20.0,NaN,living,NaN,12/8/06,NaN,0.000314,Shreve’s oak,Quercus parvula
3,SEQUSE,582311.77,4096662.27,FERP3,1420.0,NaN,living,NaN,12/8/06,NaN,1.583677,Coast redwood,Sequoia sempervirens
5,QUERPA,582314.71,4096667.18,FERP3,NaN,NaN,NaN,NaN,NaN,NaN,NaN,Shreve’s oak,Quercus parvula
...,...,...,...,...,...,...,...,...,...,...,...,...,...
51011,PSEUME,582677.22,4096891.09,FERP2,34.0,<NA>,living,NaN,7/23/14,NaN,<NA>,Douglas-fir,Pseudotsuga menziesii
51012,QUERPA,582671.76,4096867.51,FERP2,44.0,<NA>,living,NaN,7/25/14,NaN,<NA>,Shreve’s oak,Quercus parvula
51013,SEQUSE,582728.73,4096926.40,FERP2,11.0,<NA>,living,leaning,8/12/14,NaN,<NA>,Coast redwood,Sequoia sempervirens
51014,LITHDE,582651.49,4096603.23,FERP3,NaN,NaN,NaN,NaN,NaN,NaN,NaN,Tanoak,Notholithocarpus densiflorus


In [22]:
# Manual list of tree species to filter

tree_species = [
    "Shreve’s oak", "Douglas-fir", "Coast redwood", "Tanoak",
    "California hazelnut", "Coast live oak", "Madrone", "California Bay",
    "Ponderosa pine", "Knobcone pine", "Big-leaf maple", "Yellow willow",
    "Loquat", "Blue-gum eucalyptus"
]

ferp_trees_df = ferp_trees_df[ferp_trees_df['common_name'].isin(tree_species)]

len(ferp_trees_df)

31689

In [23]:
import pandas as pd
from pygbif import species
import time

# 1. Copy and normalize your scientific names

ferp_trees_df['sciname'] = (
    ferp_trees_df['scientific_name']
      .str.strip()
      .str.split()
      .str[:2]
      .str.join(' ')
      .str.lower()
)

# 2. Prepare a cache dict + manual overrides for the few bad names
taxon_cache = {}
name_overrides = {
    # 'quercus ×deamii': 'quercus deamii',  # drop the hybrid sign
    # 'salix l.':          'Salix L.',         # look up the genus  
    # 'unknown':           None             # skip entirely
}

# 3. Iterate unique names and query GBIF (with overrides)
for name in ferp_trees_df['sciname'].dropna().unique():
    if name in name_overrides:
        override = name_overrides[name]
        # if override is None, we know it’s “unknown” → leave as None
        if override is None:
            taxon_cache[name] = None
            continue
        lookup_name = override
    else:
        lookup_name = name

    try:
        res = species.name_backbone(name=lookup_name)
        key = res.get('usageKey') or None
    except Exception:
        key = None

    taxon_cache[name] = key
    time.sleep(0.1)  # be polite to GBIF

# 4. Map back onto your DataFrame
ferp_trees_df['gbif_taxon_key'] = ferp_trees_df['sciname'].map(taxon_cache)

# 5. Check what still didn’t resolve
unmapped = sorted(x for x, k in taxon_cache.items() if k is None)
print(f"Unmapped names ({len(unmapped)}):", unmapped)


/var/folders/qx/bpj16cl90cq20swjd4j79cdh0000gn/T/ipykernel_56129/1918759830.py:7: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  ferp_trees_df['sciname'] = (


Unmapped names (0): []


/var/folders/qx/bpj16cl90cq20swjd4j79cdh0000gn/T/ipykernel_56129/1918759830.py:46: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  ferp_trees_df['gbif_taxon_key'] = ferp_trees_df['sciname'].map(taxon_cache)


In [24]:
(ferp_trees_df.gbif_taxon_key.isna().sum())/len(ferp_trees_df)

np.float64(0.0)

All of these have been successfully mapped to their gbif taxon id.

Now let's remove unnecessary columns.

In [25]:
ferp_trees_df.columns

Index(['code6', 'east_UTM', 'north_UTM', 'survey_used', 'dsh_mm', 'dshm_mm',
       'status', 'condition', 'date', 'multi', 'basalarea_m2', 'common_name',
       'scientific_name', 'sciname', 'gbif_taxon_key'],
      dtype='object')

In [26]:
ferp_trees_df.isna().sum()

code6                  0
east_UTM               0
north_UTM              0
survey_used            0
dsh_mm             27134
dshm_mm            31689
status             27121
condition          31349
date               27134
multi              31027
basalarea_m2       27262
common_name            0
scientific_name        0
sciname                0
gbif_taxon_key         0
dtype: int64

In [27]:
ferp_trees_df

,code6,east_UTM,north_UTM,survey_used,dsh_mm,dshm_mm,status,condition,date,multi,basalarea_m2,common_name,scientific_name,sciname,gbif_taxon_key
0,QUERPA,582309.51,4096655.62,FERP3,31.0,NaN,living,NaN,12/8/06,NaN,0.000755,Shreve’s oak,Quercus parvula,quercus parvula,2878124
1,PSEUME,582307.45,4096655.65,FERP3,378.0,NaN,living,NaN,12/8/06,NaN,0.112221,Douglas-fir,Pseudotsuga menziesii,pseudotsuga menziesii,2685796
2,QUERPA,582307.51,4096656.25,FERP3,20.0,NaN,living,NaN,12/8/06,NaN,0.000314,Shreve’s oak,Quercus parvula,quercus parvula,2878124
3,SEQUSE,582311.77,4096662.27,FERP3,1420.0,NaN,living,NaN,12/8/06,NaN,1.583677,Coast redwood,Sequoia sempervirens,sequoia sempervirens,2683909
5,QUERPA,582314.71,4096667.18,FERP3,NaN,NaN,NaN,NaN,NaN,NaN,NaN,Shreve’s oak,Quercus parvula,quercus parvula,2878124
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
51011,PSEUME,582677.22,4096891.09,FERP2,34.0,<NA>,living,NaN,7/23/14,NaN,<NA>,Douglas-fir,Pseudotsuga menziesii,pseudotsuga menziesii,2685796
51012,QUERPA,582671.76,4096867.51,FERP2,44.0,<NA>,living,NaN,7/25/14,NaN,<NA>,Shreve’s oak,Quercus parvula,quercus parvula,2878124
51013,SEQUSE,582728.73,4096926.40,FERP2,11.0,<NA>,living,leaning,8/12/14,NaN,<NA>,Coast redwood,Sequoia sempervirens,sequoia sempervirens,2683909
51014,LITHDE,582651.49,4096603.23,FERP3,NaN,NaN,NaN,NaN,NaN,NaN,NaN,Tanoak,Notholithocarpus densiflorus,notholithocarpus densiflorus,3906623


## Merge OFO and FERP

Now lets combine the OFO and FERP datasets.

In [39]:
import pandas as pd
import geopandas as gpd
import numpy as np
from shapely.geometry import Point

# ---- 1. Copy original pandas DataFrames ----
ferp = ferp_trees_df.copy()
ofo = ofo_trees_df.copy()

# ---- 2. Convert FERP to GeoDataFrame using UTM coords ----
ferp['geometry'] = gpd.points_from_xy(ferp['east_UTM'], ferp['north_UTM'])
ferp_gdf = gpd.GeoDataFrame(ferp, geometry='geometry',
                            crs="+proj=utm +zone=10 +datum=WGS84 +units=m +no_defs")

# ---- 3. Reproject FERP to WGS84 to match OFO ----
ferp_gdf = ferp_gdf.to_crs(epsg=4326)

# ---- 4. Convert OFO to GeoDataFrame ----
ofo_gdf = gpd.GeoDataFrame(ofo, geometry='geometry', crs="EPSG:4326")

# ---- 5. Add standardized measurements ----
ferp_gdf['dbh_cm'] = ferp_gdf['dsh_mm'] / 10
ferp_gdf['height_m'] = np.nan
ferp_gdf['crown_position'] = np.nan

ofo_gdf['dbh_cm'] = ofo_gdf['dbh']
ofo_gdf['height_m'] = ofo_gdf['height']

# ---- 6. Add source and plot_id ----
ferp_gdf['source'] = 'FERP'
ferp_gdf['plot_id'] = 'F1'

ofo_gdf['source'] = 'OFO'
ofo_gdf['plot_id'] = ofo_gdf['plot_id'].astype(str).str.zfill(4)

# ---- 7. Harmonize and combine columns ----
common_cols = [
    'plot_id',
    'scientific_name', 'common_name', 'gbif_taxon_key',
    'dbh_cm', 'height_m', 'crown_position',
    'geometry', 'source'
]

ferp_final = ferp_gdf[common_cols]
ofo_final = ofo_gdf[common_cols]

combined_gdf = pd.concat([ferp_final, ofo_final], ignore_index=True)

# ---- 8. Result ----
print(f"Combined dataset has {len(combined_gdf)} records")
combined_gdf.head()


Combined dataset has 66160 records


,plot_id,scientific_name,common_name,gbif_taxon_key,dbh_cm,height_m,crown_position,geometry,source
0,F1,Quercus parvula,Shreve’s oak,2878124.0,3.1,NaN,NaN,POINT (-122.07479 37.01247),FERP
1,F1,Pseudotsuga menziesii,Douglas-fir,2685796.0,37.8,NaN,NaN,POINT (-122.07481 37.01247),FERP
2,F1,Quercus parvula,Shreve’s oak,2878124.0,2.0,NaN,NaN,POINT (-122.07481 37.01247),FERP
3,F1,Sequoia sempervirens,Coast redwood,2683909.0,142.0,NaN,NaN,POINT (-122.07476 37.01253),FERP
4,F1,Quercus parvula,Shreve’s oak,2878124.0,NaN,NaN,NaN,POINT (-122.07473 37.01257),FERP


In [40]:
combined_gdf

,plot_id,scientific_name,common_name,gbif_taxon_key,dbh_cm,height_m,crown_position,geometry,source
0,F1,Quercus parvula,Shreve’s oak,2878124.0,3.1,NaN,NaN,POINT (-122.07479 37.01247),FERP
1,F1,Pseudotsuga menziesii,Douglas-fir,2685796.0,37.8,NaN,NaN,POINT (-122.07481 37.01247),FERP
2,F1,Quercus parvula,Shreve’s oak,2878124.0,2.0,NaN,NaN,POINT (-122.07481 37.01247),FERP
3,F1,Sequoia sempervirens,Coast redwood,2683909.0,142.0,NaN,NaN,POINT (-122.07476 37.01253),FERP
4,F1,Quercus parvula,Shreve’s oak,2878124.0,NaN,NaN,NaN,POINT (-122.07473 37.01257),FERP
...,...,...,...,...,...,...,...,...,...
66155,0087,Abies concolor (Gord. & Glend.) Lindl. ex Hild...,white fir,2685580.0,5.5,3.0,NaN,POINT (-119.02263 36.96469),OFO
66156,0087,Calocedrus decurrens (Torr.) Florin,incense cedar,2683936.0,5.9,3.0,5.0,POINT (-119.02267 36.96455),OFO
66157,0087,Calocedrus decurrens (Torr.) Florin,incense cedar,2683936.0,13.1,NaN,NaN,POINT (-119.02261 36.96447),OFO
66158,0087,Frangula purshiana,NaN,8286637.0,NaN,NaN,NaN,POINT (-119.02267 36.96427),OFO


In [41]:
combined_gdf.to_file('./data/trees.gpkg', driver='GPKG')

INFO:Created 66,160 records
